In [9]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


In [10]:
df = pd.read_csv("../data/faqs.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10, 2)


,question,answer
0,How do I reset my password?,Open account settings and select Reset Password.
1,How can I contact support?,Email the support team for assistance.
2,How do I create an account?,Click Sign Up and complete the registration form.
3,What payment methods are supported?,You can pay using the supported payment options.
4,How can I delete my account?,Open account settings and follow the account d...


In [11]:
df = df.dropna(subset=["question", "answer"])

df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

df = df.drop_duplicates(subset=["question"])

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.reset_index(drop=True)

print("Clean FAQs:", len(df))

Clean FAQs: 10


In [12]:
df["text"] = (
    "Question: " + df["question"] +
    "\nAnswer: " + df["answer"]
)

chunks = df["text"].tolist()

print("Number of chunks:", len(chunks))
print("\nExample chunk:\n", chunks[0])

Number of chunks: 10

Example chunk:
 Question: How do I reset my password?
Answer: Open account settings and select Reset Password.


In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="../chroma_db")

collection = client.get_or_create_collection(
    name="faq_collection"
)

embeddings = model.encode(chunks).tolist()

collection.upsert(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings
)

print("Stored FAQs:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stored FAQs: 10


In [14]:
query = "I forgot my password and cannot log in"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

for rank, document in enumerate(
    results["documents"][0],
    start=1
):
    print(f"\nMatch {rank}:")
    print(document)


Match 1:
Question: How do I reset my password?
Answer: Open account settings and select Reset Password.

Match 2:
Question: How do I recover my username?
Answer: Use the username recovery option on the login page.

Match 3:
Question: How can I delete my account?
Answer: Open account settings and follow the account deletion instructions.
